In [ ]:
import os, json
os.makedirs("/root/.kaggle", exist_ok=True)

kaggle_credentials = {
    "username": "your kaggle username",
    "key": "your kaggle key"
}
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
import zipfile
with zipfile.ZipFile("plantvillage-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("plantvillage")
print("Dataset ready!")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:51<00:00, 42.8MB/s]

Dataset ready!


In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
print(model)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 108MB/s] 


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [4]:
# Check what the classifier looks like right now
print("Original classifier:")
print(model.classifier)
print(f"\nOriginal output size: 1000 classes")

# Replace the final Linear layer
num_classes = 38
model.classifier[1] = nn.Linear(in_features=1280, out_features=num_classes)

print("\nNew classifier:")
print(model.classifier)
print(f"\nNew output size: {num_classes} classes")

Original classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)

Original output size: 1000 classes

New classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=38, bias=True)
)

New output size: 38 classes


In [5]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# Count trainable vs frozen parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters: {frozen:,}")
print(f"\nOnly {trainable/total*100:.1f}% of the model will be trained today")

Total parameters: 4,056,226
Trainable parameters: 48,678
Frozen parameters: 4,007,548

Only 1.2% of the model will be trained today


In [6]:
# Move model to GPU
model = model.to(device)
print(f"Model moved to: {device}")

# Create a fake batch to test
fake_batch = torch.randn(4, 3, 224, 224).to(device)
print(f"\nFake input shape: {fake_batch.shape}")

# Forward pass
with torch.no_grad():
    output = model(fake_batch)

print(f"Output shape: {output.shape}")
print(f"Output values (first image, first 5 scores):")
print(output[0][:5])
print(f"\nThese 38 numbers are raw scores - highest = predicted class")

Model moved to: cuda

Fake input shape: torch.Size([4, 3, 224, 224])
Output shape: torch.Size([4, 38])
Output values (first image, first 5 scores):
tensor([ 0.0512, -0.4075, -0.0525, -0.2011,  0.1343], device='cuda:0')

These 38 numbers are raw scores - highest = predicted class


In [7]:
from torchvision.models import efficientnet_b0

# Count parameters nicely
def model_summary(model):
    print("="*50)
    print("MODEL SUMMARY")
    print("="*50)
    print(f"\nArchitecture: EfficientNet-B0")
    print(f"Input shape:  [batch, 3, 224, 224]")
    print(f"Output shape: [batch, 38]")
    print(f"\nTotal parameters:     {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Frozen parameters:    {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}")
    print(f"\nDevice: {next(model.parameters()).device}")
    print(f"\nClassifier layer:")
    print(f"  {model.classifier}")
    print("="*50)

model_summary(model)

MODEL SUMMARY

Architecture: EfficientNet-B0
Input shape:  [batch, 3, 224, 224]
Output shape: [batch, 38]

Total parameters:     4,056,226
Trainable parameters: 48,678
Frozen parameters:    4,007,548

Device: cuda:0

Classifier layer:
  Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=38, bias=True)
)
